In [6]:
# Sample an SE(3) matrix near the identity.
    #
    # Rotation part:
    #   Use the SO(3) sampler to generate a rotation whose rotation angle
    #   is bounded by theta0.
    #
    # Translation part:
    #   Sample x, y, z independently in the interval [a, b].
    #
    # Output:
    #   q : quaternion describing the sampled rotation
    #   R : 3x3 rotation matrix
    #   t : translation vector [x, y, z]
    #   G : 4x4 homogeneous SE(3) matrix

    # Check that the translation interval is valid.
    # We need a <= b so that [a,b] is a proper interval.

In [2]:
import numpy as np


In [3]:

def normalize(q: np.ndarray) -> np.ndarray:
    """Normalize a vector to unit length."""
    return q / np.linalg.norm(q)

def quat_to_R(q: np.ndarray) -> np.ndarray:
    """
    Convert a unit quaternion q = [w, x, y, z] to a 3x3 rotation matrix.
    """
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z),   2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),       1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),       2*(y*z + x*w),     1 - 2*(x*x + y*y)]
    ])

def sample_bounded_SO3(theta0: float, max_tries: int = 1_000_000) -> tuple[np.ndarray, np.ndarray]:
    """
    Sample Haar-uniform on SO(3), conditioned on rotation angle <= theta0.
    Returns quaternion q and rotation matrix R.
    """
    if not (0.0 < theta0 <= np.pi):
        raise ValueError("theta0 must satisfy 0 < theta0 <= pi.")

    c = np.cos(theta0 / 2.0)

    for _ in range(max_tries):
        # Uniform sample on S^3
        q = np.random.normal(size=4)
        q = normalize(q)

        # Fix double cover: q and -q are the same rotation
        if q[0] < 0:
            q = -q

        # Accept only if rotation angle <= theta0
        if q[0] >= c:
            R = quat_to_R(q)
            return q, R

    raise RuntimeError("Failed to sample within max_tries.")

def sample_se3(theta0: float, a: float, b: float):
    """
    Sample an SE(3) matrix with:
      - rotation angle in [0, theta0]
      - translations x, y, z in [a, b]

    Returns:
        q      : quaternion for the sampled rotation
        R      : 3x3 rotation matrix
        t      : translation vector [x, y, z]
        G      : 4x4 SE(3) matrix
    """
    if a > b:
        raise ValueError("Need a <= b.")

    # Sample bounded rotation
    q, R = sample_bounded_SO3(theta0)

    # Sample bounded translation
    x = np.random.uniform(a, b)
    y = np.random.uniform(a, b)
    z = np.random.uniform(a, b)
    t = np.array([x, y, z])

    # Build homogeneous SE(3) matrix
    G = np.eye(4)
    G[:3, :3] = R
    G[:3, 3] = t

    return q, R, t, G

In [4]:
def is_se3(G: np.ndarray, tol: float = 1e-10) -> bool:
    """
    Check whether a 4x4 matrix is in SE(3).
    """
    if G.shape != (4, 4):
        return False

    # Last row must be [0,0,0,1]
    if not np.allclose(G[3], np.array([0.0, 0.0, 0.0, 1.0]), atol=tol):
        return False

    R = G[:3, :3]

    # Check R is orthogonal
    if not np.allclose(R.T @ R, np.eye(3), atol=tol):
        return False

    # Check det(R)=1
    if not np.isclose(np.linalg.det(R), 1.0, atol=tol):
        return False

    return True

In [5]:
q, R, t, G = sample_se3(theta0=np.pi/4, a=-2.0, b=2.0)

print("Quaternion q:")
print(q)
print("\nRotation matrix R:")
print(R)
print("\nTranslation t:")
print(t)
print("\nSE(3) matrix G:")
print(G)
print("\nIs SE(3)?", is_se3(G))

Quaternion q:
[ 0.94418711 -0.00478219 -0.01208866  0.329153  ]

Rotation matrix R:
[[ 0.78302433 -0.62144842 -0.02597607]
 [ 0.62167966  0.78327087  0.00107252]
 [ 0.01967978 -0.01698861  0.99966199]]

Translation t:
[-1.4909859   0.62477301  0.68883575]

SE(3) matrix G:
[[ 7.83024335e-01 -6.21448417e-01 -2.59760672e-02 -1.49098590e+00]
 [ 6.21679658e-01  7.83270868e-01  1.07252478e-03  6.24773012e-01]
 [ 1.96797779e-02 -1.69886056e-02  9.99661990e-01  6.88835750e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

Is SE(3)? True
